<a href="https://colab.research.google.com/github/Zyu-Peng/learning_code/blob/AF2/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.6.0: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [ ]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title Install dependencies
%%bash -s $use_amber $use_templates $python_version

set -e

USE_AMBER=$1
USE_TEMPLATES=$2
PYTHON_VERSION=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  # hack to fix TF crash
  rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so
  touch COLABFOLD_READY
fi

# Download params (~1min)
python -m colabfold.download

# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
    bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniforge3-25.3.1-0-Linux-x86_64.sh
    conda config --set auto_update_conda false
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python="${PYTHON_VERSION}" 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=8.2.0 python="${PYTHON_VERSION}" pdbfixer 2>&1 1>/dev/null
  touch AMBER_READY
fi

In [ ]:
import sys
import re
import os
import glob
from pathlib import Path
from colabfold.batch import run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging

# --- 新增：进度管理功能 ---
checkpoint_file = Path(result_dir) / "processed_tasks.log"

def load_checkpoint():
    """读取已完成的任务列表"""
    if checkpoint_file.exists():
        with open(checkpoint_file, "r") as f:
            return set(line.strip() for line in f if line.strip())
    return set()

def update_checkpoint(seq_name):
    """记录新完成的任务"""
    with open(checkpoint_file, "a") as f:
        f.write(f"{seq_name}\n")

# --- 修改后的解析函数 ---
def parse_all_sequences(input_path):
    queries = []
    input_path = Path(input_path)

    # 加载已跳过的进度
    done_set = load_checkpoint()

    if input_path.is_dir():
        fasta_files = list(input_path.glob("*.fasta")) + list(input_path.glob("*.fa"])
    else:
        fasta_files = [input_path]

    for fasta_file in fasta_files:
        with open(fasta_file, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]

        current_name, current_seq = None, []
        for line in lines:
            if line.startswith('>'):
                if current_name and current_seq:
                    clean_name = re.sub(r'[^\w\-_\.]', '_', current_name.split('|')[0].strip())
                    # 检查是否已处理
                    if clean_name not in done_set:
                        queries.append((clean_name, ''.join(current_seq), None, None))
                current_name = line[1:]
                current_seq = []
            else:
                current_seq.append(re.sub(r'[^A-Za-z]', '', line).upper())

        # 处理最后一个序列
        if current_name and current_seq:
            clean_name = re.sub(r'[^\w\-_\.]', '_', current_name.split('|')[0].strip())
            if clean_name not in done_set:
                queries.append((clean_name, ''.join(current_seq), None, None))

    return queries, False

# --- 修改后的清理函数 ---
def clean_up_results(result_dir, seq_name):
    """
    针对单个序列进行清理，并更新进度
    """
    result_path = Path(result_dir)
    found = False

    # 查找该序列对应的模型文件
    for file in list(result_path.glob(f"{seq_name}*.pdb")):
        if "model_1" in file.name:
            new_name = result_path / f"{seq_name}.pdb"
            if new_name.exists(): new_name.unlink()
            file.rename(new_name)
            found = True
        else:
            file.unlink() # 删除其他 model 的 pdb

    # 删除该序列产生的其他冗余文件 (json, a3m, bib 等)
    for extra in list(result_path.glob(f"{seq_name}*")):
        if extra.is_file() and extra.suffix != ".pdb":
            extra.unlink()

    if found:
        update_checkpoint(seq_name) # 只有成功生成 PDB 后才记录进度
        print(f"已完成并记录: {seq_name}.pdb")

# --- 主逻辑循环 ---
setup_logging(Path(result_dir).joinpath("log.txt"))
queries, is_complex = parse_all_sequences(input_dir)

if not queries:
    print("所有序列均已处理完成，或输入目录为空。")
else:
    for q in queries:
        current_query = [q] # 每次只跑一个序列，方便中途停止时保留进度
        seq_name = q[0]

        print(f"正在处理: {seq_name}...")

        run(
            queries=current_query,
            result_dir=result_dir,
            use_templates=use_templates,
            num_relax=num_relax,
            msa_mode=msa_mode,
            model_type="alphafold2",
            num_models=1,
            num_recycles=num_recycles,
            model_order=[1],
            is_complex=is_complex,
            data_dir=default_data_dir,
            keep_existing_results=do_not_overwrite_results,
            stop_at_score=stop_at_score,
            zip_results=False,
        )

        # 运行完一个，清理一个，记录一个
        clean_up_results(result_dir, seq_name)

print(f"任务运行结束。进度保存在: {checkpoint_file}")

2026-03-16 16:39:23,941 Running on GPU
2026-03-16 16:39:24,110 Found 5 citations for tools or databases
2026-03-16 16:39:24,111 Query 1/966: seq_1 (length 40)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:00 remaining: 00:00]


2026-03-16 16:39:37,340 Padding length to 50
2026-03-16 16:40:01,276 alphafold2_model_1_seed_000 recycle=0 pLDDT=62.2
2026-03-16 16:40:23,858 alphafold2_model_1_seed_000 recycle=1 pLDDT=68.4 tol=1.35
2026-03-16 16:40:28,302 alphafold2_model_1_seed_000 recycle=2 pLDDT=72.8 tol=0.267
2026-03-16 16:40:32,777 alphafold2_model_1_seed_000 recycle=3 pLDDT=70.6 tol=0.496
2026-03-16 16:40:32,778 alphafold2_model_1_seed_000 took 55.4s (3 recycles)
2026-03-16 16:40:32,791 reranking models by 'plddt' metric
2026-03-16 16:40:32,792 rank_001_alphafold2_model_1_seed_000 pLDDT=70.6
2026-03-16 16:40:33,020 Query 2/966: seq_2 (length 26)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-03-16 16:40:35,672 Padding length to 50
2026-03-16 16:40:40,287 alphafold2_model_1_seed_000 recycle=0 pLDDT=90.8
2026-03-16 16:40:44,778 alphafold2_model_1_seed_000 recycle=1 pLDDT=90.7 tol=0.068
2026-03-16 16:40:49,290 alphafold2_model_1_seed_000 recycle=2 pLDDT=90.9 tol=0.0306
2026-03-16 16:40:53,829 alphafold2_model_1_seed_000 recycle=3 pLDDT=91 tol=0.017
2026-03-16 16:40:53,830 alphafold2_model_1_seed_000 took 18.2s (3 recycles)
2026-03-16 16:40:53,844 reranking models by 'plddt' metric
2026-03-16 16:40:53,845 rank_001_alphafold2_model_1_seed_000 pLDDT=91
2026-03-16 16:40:54,151 Query 3/966: seq_3 (length 39)


...

2026-03-16 19:22:27,361 Sleeping for 8s. Reason: PENDING


RUNNING:   5%|▌         | 8/150 [elapsed: 00:09 remaining: 02:40]

2026-03-16 19:22:35,881 Sleeping for 10s. Reason: RUNNING


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:20 remaining: 00:00]


2026-03-16 19:22:48,626 Padding length to 50
2026-03-16 19:22:53,149 alphafold2_model_1_seed_000 recycle=0 pLDDT=51.2
